In [1]:
import numpy as np
import torch
from torch import nn
import pandas as pd
import os
import sys
import copy
import matplotlib.pyplot as plt

PATH = ""
# PATH = "storage"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Data Setup

In [2]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from reetoolbox.dataloaders import *

import pandas as pd
import numpy as np
import copy
import time
from reetoolbox.metrics import adversarial_pr_auc, adversarial_roc_auc, adversarial_accuracy
from reetoolbox.optimisers import PGD, StochasticSearch
from reetoolbox.image_evaluator import Evaluator

import matplotlib.pyplot as plt
import seaborn as sns

import os
from huggingface_hub import login, hf_hub_download
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from timm.layers import SwiGLUPacked

In [3]:
# dataset_name = "NCT"
# batch_size           = 8
# mode                 = 'tum_vs_all'
# test_multiplier      = 0.012
# test_root            = os.path.join(PATH, 'data', 'NCT', 'CRC-VAL-HE-7K')
# mode                 = 'tum_vs_all'

# test_data, test_loader = build_dataset(
#     NCTDataSet,
#     root_dir=test_root,
#     batch_size=batch_size,
#     test_multiplier=test_multiplier,
#     transform=transforms.Compose([transforms.ToTensor()]),
#     classification_mode=mode,
#     shuffle=False
# )

# print(f"Test batches: {len(test_loader)}")

In [4]:
# dataset_name = "PanNuke"
# root_dir = os.path.join(PATH, 'data', dataset_name)
# batch_size = 8
# test_multiplier = 0.035

# test_data, test_loader = build_dataset(
#     PanNukeDataset,
#     root_dir=root_dir,
#     batch_size=batch_size,
#     test_multiplier=test_multiplier,
#     transform=transforms.Compose([transforms.ToTensor()]),
#     folds=(3,),
#     min_positive=5,
#     shuffle=False
# )

# print(f"Test batches: {len(test_loader)}")

In [5]:
# dataset_name = "PANDA"

# panda_root = os.path.join(PATH, 'data', dataset_name)
# batch_size = 8
# test_multipler = 0.012

# test_data, test_loader = build_dataset(
#     PandaDataset, 
#     root_dir = panda_root, 
#     batch_size = batch_size, 
#     test_multiplier = test_multipler, 
#     transform=transforms.Compose([transforms.ToTensor()]), 
#     split='test'
# )

# print(f"Test batches: {len(test_loader)}")

In [6]:
dataset_name = "PatchCamelyon"

patch_root    = os.path.join(PATH, 'data', 'PatchCamelyon')
batch_size    = 8
test_multiplier = 0.00264

test_data, test_loader = build_dataset(
    PatchCamelyonDataset,
    root_dir = patch_root,
    batch_size = batch_size,
    test_multiplier = test_multiplier,
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ]),
    split = 'test'
)

print(f"Test batches: {len(test_loader)}")

Test batches: 11


In [7]:
import torch

def count_class_distribution(dataloader):
    counts = None
    for _, labels, _ in dataloader:
        # Ensure labels are 1D torch.Tensor
        bincount = torch.bincount(labels, minlength=labels.max().item() + 1)
        if counts is None:
            counts = bincount
        else:
            # If batch has more classes than previous batches, expand counts
            if bincount.numel() > counts.numel():
                counts = torch.cat([counts, torch.zeros(bincount.numel() - counts.numel(), dtype=counts.dtype)])
            counts[:bincount.numel()] += bincount
    return counts

test_counts = count_class_distribution(test_loader)

print(f"Test: {test_counts}")

Test: tensor([43, 43])


In [8]:
github_path = os.path.join(PATH, "Models", "GitHub", "EXAONEPath")
sys.path.append(github_path)
from vision_transformer import VisionTransformer
from huggingface_hub import login, hf_hub_download

In [9]:
from reetoolbox.loadmodels import *
from reetoolbox.eval_funcs import *
# — login once per session —
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Please set HF_TOKEN in your environment")
login(token=hf_token, add_to_git_credential=True)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Evaluation

In [10]:
from reetoolbox.transforms import (
    PixelTransform,          # → eval_pixel_optimiser_params, eval_pixel_transform_params
    StainTransform,          # → eval_stain_optimiser_params, eval_stain_transform_params
    MeanTransform,           # → eval_mean_optimiser_params, eval_mean_transform_params
    RotateTransform,         # → eval_rotate_optimiser_params, eval_rotate_transform_params
    CropTransform,           # → eval_crop_optimiser_params, eval_crop_transform_params
    BlurTransform,           # → eval_blur_optimiser_params, eval_blur_transform_params
    ZoomInTransform,         # → eval_zoom_in_optimiser_params, eval_zoom_in_transform_params
    ZoomOutTransform,        # → eval_zoom_out_optimiser_params, eval_zoom_out_transform_params
    HEDTransform,            # → eval_hed_optimiser_params, eval_hed_transform_params
    RandomStainTransform,    # → eval_random_stain_optimiser_params, eval_random_stain_transform_params
    JPEGTransform,           # → eval_jpeg_optimiser_params, eval_jpeg_transform_params
)

from reetoolbox.constants import (
    eval_pixel_optimiser_params,        eval_pixel_transform_params,
    eval_stain_optimiser_params,        eval_stain_transform_params,
    eval_mean_optimiser_params,         eval_mean_transform_params,
    eval_crop_optimiser_params,         eval_crop_transform_params,
    eval_rotate_optimiser_params,       eval_rotate_transform_params,
    eval_blur_optimiser_params,         eval_blur_transform_params,
    eval_zoom_in_optimiser_params,      eval_zoom_in_transform_params,
    eval_zoom_out_optimiser_params,     eval_zoom_out_transform_params,
    eval_hed_optimiser_params,          eval_hed_transform_params,
    eval_random_stain_optimiser_params, eval_random_stain_transform_params,
    eval_jpeg_optimiser_params,         eval_jpeg_transform_params,
)

perturbations = {
    "pixel": {
        "TransformOptimiser": PGD,
        "Transform": PixelTransform,
        "optimiser_params": eval_pixel_optimiser_params,
        "transform_params": eval_pixel_transform_params,
    },
    "mean": {
        "TransformOptimiser": PGD,
        "Transform": MeanTransform,
        "optimiser_params": eval_mean_optimiser_params,
        "transform_params": eval_mean_transform_params,
    },
    "random_stain": {
        "TransformOptimiser": StochasticSearch,
        "Transform": RandomStainTransform,
        "optimiser_params": eval_random_stain_optimiser_params,
        "transform_params": eval_random_stain_transform_params,
    },
    "jpeg": {
        "TransformOptimiser": StochasticSearch,
        "Transform": JPEGTransform,
        "optimiser_params": eval_jpeg_optimiser_params,
        "transform_params": eval_jpeg_transform_params,
    },
    "blur": {
        "TransformOptimiser": StochasticSearch,
        "Transform": BlurTransform,
        "optimiser_params": eval_blur_optimiser_params,
        "transform_params": eval_blur_transform_params,
    },
    "rotate": {
        "TransformOptimiser": StochasticSearch,
        "Transform": RotateTransform,
        "optimiser_params": eval_rotate_optimiser_params,
        "transform_params": eval_rotate_transform_params,
    },
    "zoom_in": {
        "TransformOptimiser": StochasticSearch,
        "Transform": ZoomInTransform,
        "optimiser_params": eval_zoom_in_optimiser_params,
        "transform_params": eval_zoom_in_transform_params,
    },
    "zoom_out": {
        "TransformOptimiser": StochasticSearch,
        "Transform": ZoomOutTransform,
        "optimiser_params": eval_zoom_out_optimiser_params,
        "transform_params": eval_zoom_out_transform_params,
    },
    "crop": {
        "TransformOptimiser": StochasticSearch,
        "Transform": CropTransform,
        "optimiser_params": eval_crop_optimiser_params,
        "transform_params": eval_crop_transform_params,
    },
}

sweep_params = {
    "pixel":        "C",
    "mean":         "C",
    "random_stain": "weights",
    "jpeg":         "quality",
    "blur":         "sigma",      
    "rotate":       "angle",
    "zoom_in":      "scale",
    "zoom_out":     "scale",
    "crop":         "height",     
    "hed":          "alpha",      
    "stain":        "C",          
}

# Number of steps for each sweep
n_pixel_steps      = 20
n_mean_steps       = 20
n_randomstain_steps= 25
n_jpeg_steps       = 11
n_blur_steps       = 21
n_rotate_steps     = 37
n_zoomin_steps     = 20
n_zoomout_steps    = 20
n_crop_steps       = 20
n_hed_steps        = 21
n_stain_steps      = 20

# Example: For 224x224 images, crop from 224 (no crop) to 100 (strong crop)
sweep_ranges = {
    "pixel":        (0.00, 0.05),
    "mean":         (0.00, 7.5),
    "random_stain": (0.00, 0.5),
    "jpeg":         (90, 100),
    "blur":         (1, 7),
    "rotate":       (0, 360),
    "zoom_in":      (1.0, 3.0),
    "zoom_out":     (1.0, 0.05),
    "crop":         (224, 50),
}

step_sizes = {
    "pixel":        (sweep_ranges["pixel"][1] - sweep_ranges["pixel"][0]) / (n_pixel_steps-1),
    "mean":         (sweep_ranges["mean"][1] - sweep_ranges["mean"][0]) / (n_mean_steps-1),
    "random_stain": (sweep_ranges["random_stain"][1] - sweep_ranges["random_stain"][0]) / (n_randomstain_steps-1),
    "jpeg":         (sweep_ranges["jpeg"][1] - sweep_ranges["jpeg"][0]) / (n_jpeg_steps-1),
    "blur":         (sweep_ranges["blur"][1] - sweep_ranges["blur"][0]) / (n_blur_steps-1),
    "rotate":       (sweep_ranges["rotate"][1] - sweep_ranges["rotate"][0]) / (n_rotate_steps-1),
    "zoom_in":      (sweep_ranges["zoom_in"][1] - sweep_ranges["zoom_in"][0]) / (n_zoomin_steps-1),
    "zoom_out":     -(sweep_ranges["zoom_out"][0] - sweep_ranges["zoom_out"][1]) / (n_zoomout_steps-1),
    "crop":         -(sweep_ranges["crop"][0] - sweep_ranges["crop"][1]) / (n_crop_steps-1),   # decreasing from 224 to 100
    # "hed":          (sweep_ranges["hed"][1] - sweep_ranges["hed"][0]) / (n_hed_steps-1),
    # "stain":        (sweep_ranges["stain"][1] - sweep_ranges["stain"][0]) / (n_stain_steps-1),
}

In [11]:
runs = 3  # number of runs

In [ ]:
model_evals = [
    {
        "model_name": "ResNet18",
        "load_func": lambda: torch.load(os.path.join(PATH, "Models", dataset_name, f"resnet18_{dataset_name}.pth"), weights_only=False),
        "weight_path": None,  # already loaded with torch.load
        "test_loader_key": "resnet",
        "output_subdir": "ResNet18",
    },
    {
        "model_name": "ResNet50",
        "load_func": lambda: torch.load(os.path.join(PATH, "Models", dataset_name, f"resnet50_{dataset_name}.pth"), weights_only=False),
        "weight_path": None,  # already loaded with torch.load
        "test_loader_key": "resnet",
        "output_subdir": "ResNet50",
    },
# Group A
    {
        "model_name": "UNI",
        "load_func": lambda: load_uni(n_classes=2)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"uni_{dataset_name}.pth"),
        "test_loader_key": "UNI",
        "output_subdir": "UNI",
    },
    {
        "model_name": "Phikon v2",
        "load_func": lambda: load_phikon_v2(n_classes=2, device=device)[1],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"phikon_v2_{dataset_name}.pth"),
        "test_loader_key": "Phikon v2",
        "output_subdir": "Phikon v2",
    },
    {
        "model_name": "Hibou-L",
        "load_func": lambda: load_hibou_l(n_classes=2, device=device)[1],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"hibou_l_{dataset_name}.pth"),
        "test_loader_key": "Hibou-L",
        "output_subdir": "Hibou-L",
    },
    {
        "model_name": "Virchow",
        "load_func": lambda: load_virchow(n_classes=2, device=device)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"virchow_{dataset_name}.pth"),
        "test_loader_key": "Virchow",
        "output_subdir": "Virchow",
    },
    {
        "model_name": "Virchow2",
        "load_func": lambda: load_virchow2(n_classes=2, device=device)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"virchow2_{dataset_name}.pth"),
        "test_loader_key": "Virchow2",
        "output_subdir": "Virchow2",
    },
    {
        "model_name": "H-Optimus-0",
        "load_func": lambda: load_h_optimus0(n_classes=2, device=device)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"hoptimus0_{dataset_name}.pth"),
        "test_loader_key": "H-Optimus-0",
        "output_subdir": "H-Optimus-0",
    },

    # Group B
    {
        "model_name": "H0-mini",
        "load_func": lambda: load_h0_mini(n_classes=2, device=device)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"h0_mini_{dataset_name}.pth"),
        "test_loader_key": "H0-mini",
        "output_subdir": "H0-mini",
    },
    {
        "model_name": "Hibou-B",
        "load_func": lambda: load_hibou_b(n_classes=2, device=device)[1],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"hibou_b_{dataset_name}.pth"),
        "test_loader_key": "Hibou-B",
        "output_subdir": "Hibou-B",
    },
    {
        "model_name": "EXAONEPath",
        "load_func": lambda: load_exaonepath(n_classes=2, device=device)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"exaonepath_{dataset_name}.pth"),
        "test_loader_key": "EXAONEPath",
        "output_subdir": "EXAONEPath",
    },
    {
        "model_name": "Phikon v1",
        "load_func": lambda: load_phikon(n_classes=2, device=device)[1],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"phikon_{dataset_name}.pth"),
        "test_loader_key": "Phikon v1",
        "output_subdir": "Phikon v1",
    },
    {
        "model_name": "UNI2",
        "load_func": lambda: load_uni2(n_classes=2)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"uni2_{dataset_name}.pth"),
        "test_loader_key": "UNI2",
        "output_subdir": "UNI2",
    },
    {
        "model_name": "H-Optimus-1",
        "load_func": lambda: load_h_optimus1(n_classes=2, device=device)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"hoptimus1_{dataset_name}.pth"),
        "test_loader_key": "H-Optimus-1",
        "output_subdir": "H-Optimus-1",
    },
    {
        "model_name": "GigaPath",
        "load_func": lambda: load_gigapath(n_classes=2)[0],
        "weight_path": os.path.join(PATH, "Models", dataset_name, f"gigapath_{dataset_name}.pth"),
        "test_loader_key": "GigaPath",
        "output_subdir": "GigaPath",
    },
]

In [13]:
from huggingface_hub import login
import torch
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from torchvision import transforms

In [14]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from torchvision import transforms

def run_all_model_sweeps(
    model_evals,
    test_data,
    test_loader,
    perturbations,
    sweep_params,
    sweep_ranges,
    step_sizes,
    metric,
    runs,
    device,
    PATH,
):
    for model_eval in model_evals:
        print(f"----------------------Beginning {model_eval['model_name']} sweep----------------------")

        # a. Load the model
        model = model_eval["load_func"]()
        if model_eval["weight_path"] is not None:
            model.head.load_state_dict(torch.load(model_eval["weight_path"]))
        model = model.to(device).eval()

        # b. Custom or default perturbations
        model_perturbations = model_eval.get("perturbations", perturbations)
        out_dir = os.path.join(PATH, "Stats", dataset_name, model_eval["output_subdir"], "constraint_sweep")

        # c. Run batch sweep (pass test_data and test_loader)
        batch_sweep(
            model         = model,
            model_name    = model_eval["model_name"],
            dataset       = test_data,
            dataloader    = test_loader,
            perturbations = model_perturbations,
            sweep_params  = sweep_params,
            sweep_ranges  = sweep_ranges,
            step_sizes    = step_sizes,
            metric        = metric,
            runs          = runs,
            device        = device,
            output_dir    = out_dir,
        )

In [ ]:
run_all_model_sweeps(
    model_evals=model_evals,
    test_data=test_data,
    test_loader=test_loader,
    perturbations=perturbations,
    sweep_params=sweep_params,
    sweep_ranges=sweep_ranges,
    step_sizes=step_sizes,
    metric=adversarial_roc_auc,
    runs=3,
    device=device,
    PATH=PATH,
)

### Create plots

In [16]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params} ({total_params/1e6:.2f}M)")
    return total_params

In [17]:
model_list = []
for i in range(len(model_evals)):
    model_list.append(model_evals[i]["model_name"])
perturb_list = list(perturbations.keys())

print(model_list)

model_param_counts = {}

for model_eval in model_evals:
        model_name = model_eval["model_name"]
        model = model_eval["load_func"]()
        count = round(count_parameters(model) / 1e6, 1)
        model_param_counts[model_name] = count

model_param_counts = dict(sorted(model_param_counts.items(), key=lambda x:x[1]))
model_param_counts

['ResNet18', 'ResNet50', 'UNI', 'Phikon v2', 'Hibou-L', 'Virchow', 'Virchow2', 'H-Optimus-0', 'H0-mini', 'Hibou-B', 'EXAONEPath', 'Phikon v1', 'UNI2', 'H-Optimus-1', 'GigaPath']
Total parameters: 11177538 (11.18M)
Total parameters: 23512130 (23.51M)
Total parameters: 303352834 (303.35M)
Total parameters: 303353858 (303.35M)
Total parameters: 303661314 (303.66M)
Total parameters: 631234306 (631.23M)
Total parameters: 631244546 (631.24M)
Total parameters: 1134777346 (1134.78M)
Total parameters: 85742594 (85.74M)
Total parameters: 85742594 (85.74M)
Total parameters: 85800194 (85.80M)


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Total parameters: 86390786 (86.39M)
Total parameters: 681397250 (681.40M)
Total parameters: 1134777346 (1134.78M)
Total parameters: 1134957058 (1134.96M)


{'ResNet18': 11.2,
 'ResNet50': 23.5,
 'H0-mini': 85.7,
 'Hibou-B': 85.7,
 'EXAONEPath': 85.8,
 'Phikon v1': 86.4,
 'UNI': 303.4,
 'Phikon v2': 303.4,
 'Hibou-L': 303.7,
 'Virchow': 631.2,
 'Virchow2': 631.2,
 'UNI2': 681.4,
 'H-Optimus-0': 1134.8,
 'H-Optimus-1': 1134.8,
 'GigaPath': 1135.0}

In [18]:
for dataset_name in ["PatchCamelyon", "NCT", "PanNuke", "PANDA"]:
# for dataset_name in ["NCT"]:
    print(f"Running {dataset_name}")
    aggregate_constraint_sweeps(
        models=model_list,
        dataset_name=dataset_name,
        perturbs=perturb_list,
        base_dir=os.path.join(PATH, 'Stats', dataset_name),
        output_dir=os.path.join(PATH, 'Stats', dataset_name, 'constraint_sweep')
    )

    calculate_PPI_aggregate(model_list, perturb_list, dataset_name = dataset_name, PATH = PATH, model_param_counts=model_param_counts)
    print("")

Running PatchCamelyon
Saved aggregate plot: Stats\PatchCamelyon\constraint_sweep\pixel_aggregate.png
Saved aggregate plot: Stats\PatchCamelyon\constraint_sweep\mean_aggregate.png
Saved aggregate plot: Stats\PatchCamelyon\constraint_sweep\random_stain_aggregate.png
Saved aggregate plot: Stats\PatchCamelyon\constraint_sweep\jpeg_aggregate.png
Saved aggregate plot: Stats\PatchCamelyon\constraint_sweep\blur_aggregate.png
Saved aggregate plot: Stats\PatchCamelyon\constraint_sweep\rotate_aggregate.png
Saved aggregate plot: Stats\PatchCamelyon\constraint_sweep\zoom_in_aggregate.png
Saved aggregate plot: Stats\PatchCamelyon\constraint_sweep\zoom_out_aggregate.png
Saved aggregate plot: Stats\PatchCamelyon\constraint_sweep\crop_aggregate.png
Saved PPI mean/std tables to Stats\PatchCamelyon\constraint_sweep\ppi_aggregate.xlsx
Saved bar chart to Stats\PatchCamelyon\constraint_sweep\ppi_mean_bar.png
Saved Stats\PatchCamelyon\constraint_sweep\ppi\PPI_vs_param_pixel.png
Saved Stats\PatchCamelyon\cons